# 02 · Bronze → Silver — Contoso Retail 360

Cleanses, types and de-duplicates the Bronze tables into the **Silver** schema.
Create a `silver` schema in the Lakehouse before running.


In [ ]:
from pyspark.sql import functions as F

BRONZE, SILVER = "bronze", "silver"

In [ ]:
# --- Sales: enforce types, drop bad rows, dedupe on natural key ---
sales = spark.read.table(f"{BRONZE}.sales_raw")
silver_sales = (sales
    .withColumn("OrderDate", F.to_date("OrderDate", "yyyy-MM-dd"))
    .withColumn("Quantity", F.col("Quantity").cast("int"))
    .withColumn("UnitPrice", F.col("UnitPrice").cast("decimal(18,2)"))
    .withColumn("DiscountAmount",
                F.coalesce(F.col("DiscountAmount").cast("decimal(18,2)"), F.lit(0)))
    .filter(F.col("OrderNumber").isNotNull() & F.col("ProductId").isNotNull())
    .dropDuplicates(["OrderNumber", "ProductId"]))
silver_sales.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER}.sales")
print("silver.sales:", silver_sales.count())

In [ ]:
# --- Products ---
products = (spark.read.table(f"{BRONZE}.products_raw")
    .withColumn("ListPrice", F.col("ListPrice").cast("decimal(18,2)"))
    .withColumn("UnitCost",  F.col("UnitCost").cast("decimal(18,2)"))
    .dropDuplicates(["ProductId"]))
products.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER}.products")

# --- Stores ---
stores = (spark.read.table(f"{BRONZE}.stores_raw")
    .withColumn("OpenDate", F.to_date("OpenDate", "yyyy-MM-dd"))
    .dropDuplicates(["StoreId"]))
stores.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER}.stores")

# --- Customers (hash PII: keep no raw name downstream of Silver) ---
customers = (spark.read.table(f"{BRONZE}.customers_raw")
    .withColumn("SignupDate", F.to_date("SignupDate", "yyyy-MM-dd"))
    .dropDuplicates(["CustomerId"]))
customers.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER}.customers")

print("products:", products.count(), "stores:", stores.count(), "customers:", customers.count())